# Project Audit and Data Review

## Project context

This notebook audits the starting point for the News Sentiment and Market Connectedness Engine. The project is a portfolio research workflow for connecting financial news sentiment, market data, signal-style logs, and connectedness analysis. Phase 0 does not run scraping, trading backtests, or final connectedness modeling.

## Phase 0 objective

Phase 0 creates a clean project structure, checks whether prototype files are available, inventories legacy scripts, and documents what can be inspected safely before Phase 1.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.config import RAW_DATA_DIR, OUTPUTS_DIR, FIGURES_DIR, LEGACY_DIR
from src.ingestion import load_csv, load_json_lines_or_records, inspect_dataframe, save_audit_summary
from src.sentiment import standardize_sentiment_log, summarize_sentiment_by_company, summarize_sentiment_by_date
from src.market_data import identify_market_columns, summarize_market_data
from src.merge import inspect_merged_dataset, identify_join_keys
from src.connectedness import inspect_connectedness_inputs, describe_gfevd_requirements

AUDIT_DIR = OUTPUTS_DIR / "audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Available file inventory

The audit checks for the expected raw/prototype files inside `data/raw/`. If files were uploaded elsewhere, they should be copied into `data/raw/` before rerunning this notebook.

In [ ]:
expected_data_files = [
    "merged_data.json",
    "scraped_news.csv",
    "sentiment_log.csv",
]

file_rows = []
for file_name in expected_data_files:
    path = RAW_DATA_DIR / file_name
    file_rows.append({
        "file_name": file_name,
        "expected_path": str(path.relative_to(PROJECT_ROOT)),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
        "status": "found" if path.exists() else "missing",
        "notes": "Copied into project raw data folder" if path.exists() else "Not found during Phase 0 audit",
    })

file_inventory = pd.DataFrame(file_rows)
file_inventory.to_csv(AUDIT_DIR / "file_inventory.csv", index=False)
file_inventory

## Legacy script inventory

Legacy scripts are kept under `legacy/` so the old pipeline can be reviewed separately from new reusable modules.

In [ ]:
expected_scripts = [
    "step1_sentiment_test.py",
    "step2_scraper.py",
    "step3_stock_filter.py",
    "step4_keyword_generator.py",
    "step6_daily_sentiment.py",
    "step7_merge_data.py",
    "step8_gfevd_analysis.py",
]

script_rows = []
for file_name in expected_scripts:
    path = LEGACY_DIR / file_name
    text = path.read_text(encoding="utf-8") if path.exists() else ""
    placeholder = "not found" in text.lower()
    script_rows.append({
        "script_name": file_name,
        "path": str(path.relative_to(PROJECT_ROOT)),
        "exists": path.exists(),
        "line_count": len(text.splitlines()) if text else 0,
        "status": "placeholder" if placeholder else ("found" if path.exists() else "missing"),
        "notes": "Original script not found; placeholder preserves structure" if placeholder else "Ready for review",
    })

legacy_script_inventory = pd.DataFrame(script_rows)
legacy_script_inventory.to_csv(AUDIT_DIR / "legacy_script_inventory.csv", index=False)
legacy_script_inventory

## Load available datasets

The notebook loads any available expected files. Missing files are handled explicitly rather than replaced with fake sample results.

In [ ]:
datasets = {}
load_notes = []

for file_name in expected_data_files:
    path = RAW_DATA_DIR / file_name
    if not path.exists():
        load_notes.append({"dataset": file_name, "status": "missing", "notes": "File not available"})
        continue
    try:
        if path.suffix.lower() == ".csv":
            df = load_csv(path)
        else:
            df = load_json_lines_or_records(path)
        datasets[file_name] = df
        load_notes.append({"dataset": file_name, "status": "loaded", "notes": f"Loaded {len(df)} rows"})
    except Exception as exc:
        load_notes.append({"dataset": file_name, "status": "error", "notes": str(exc)})

pd.DataFrame(load_notes)

## Scraped news review if available

In [ ]:
scraped_news = datasets.get("scraped_news.csv", pd.DataFrame())
if scraped_news.empty:
    print("scraped_news.csv is not available for Phase 0 review.")
else:
    display(scraped_news.head())
    display(inspect_dataframe(scraped_news, "scraped_news"))

## Sentiment log review if available

In [ ]:
sentiment_log = datasets.get("sentiment_log.csv", pd.DataFrame())
if sentiment_log.empty:
    print("sentiment_log.csv is not available for Phase 0 review.")
    standardized_sentiment = pd.DataFrame()
else:
    standardized_sentiment = standardize_sentiment_log(sentiment_log)
    display(standardized_sentiment.head())
    display(inspect_dataframe(standardized_sentiment, "sentiment_log"))

## Merged data review if available

In [ ]:
merged_data = datasets.get("merged_data.json", pd.DataFrame())
if merged_data.empty:
    print("merged_data.json is not available for Phase 0 review.")
else:
    display(merged_data.head())
    display(inspect_merged_dataset(merged_data))
    display(identify_join_keys(merged_data))

## Column and schema audit

In [ ]:
schema_rows = []
for name, df in datasets.items():
    if df.empty:
        continue
    summary = inspect_dataframe(df, name)
    schema_rows.append(summary)

if not schema_rows:
    schema_rows = [{
        "dataset": "no_loaded_data",
        "rows": 0,
        "columns": 0,
        "column_names": "",
        "missing_values_total": 0,
        "duplicate_rows": 0,
        "date_column": "",
        "start_date": pd.NaT,
        "end_date": pd.NaT,
        "valid_date_rows": 0,
    }]

schema_summary = save_audit_summary(schema_rows, AUDIT_DIR / "data_schema_summary.csv")
schema_summary

## Date coverage audit

In [ ]:
coverage_rows = []
for _, row in schema_summary.iterrows():
    coverage_rows.append({
        "dataset": row["dataset"],
        "date_column": row.get("date_column", ""),
        "start_date": row.get("start_date", pd.NaT),
        "end_date": row.get("end_date", pd.NaT),
        "valid_date_rows": row.get("valid_date_rows", 0),
        "status": "available" if row.get("valid_date_rows", 0) else "not_available",
        "notes": "Date coverage detected" if row.get("valid_date_rows", 0) else "No date coverage available in Phase 0",
    })

coverage_summary = pd.DataFrame(coverage_rows)
coverage_summary.to_csv(AUDIT_DIR / "data_coverage_summary.csv", index=False)
coverage_summary

## Company coverage audit

In [ ]:
company_rows = []
for name, df in datasets.items():
    lower_map = {col: str(col).lower() for col in df.columns}
    key_cols = [col for col, lower in lower_map.items() if lower in {"company", "ticker", "symbol", "stock"}]
    if key_cols:
        key_col = key_cols[0]
        counts = df[key_col].fillna("missing").value_counts().reset_index()
        counts.columns = ["company_key", "record_count"]
        for _, count_row in counts.iterrows():
            company_rows.append({
                "dataset": name,
                "key_column": key_col,
                "company_key": count_row["company_key"],
                "record_count": int(count_row["record_count"]),
            })

if not company_rows:
    company_rows = [{
        "dataset": "no_loaded_data",
        "key_column": "",
        "company_key": "not_available",
        "record_count": 0,
    }]

company_coverage = pd.DataFrame(company_rows)
company_coverage.to_csv(AUDIT_DIR / "company_coverage_summary.csv", index=False)
company_coverage.head(20)

## Initial sentiment distribution

In [ ]:
if standardized_sentiment.empty:
    print("No sentiment log available, so sentiment distribution was not plotted.")
else:
    sentiment_cols = [col for col in standardized_sentiment.columns if "sentiment" in col]
    print("Sentiment columns:", sentiment_cols)

## Market column availability

In [ ]:
market_audits = []
for name, df in datasets.items():
    market_audits.append({"dataset": name, **identify_market_columns(df)})

market_column_availability = pd.DataFrame(market_audits) if market_audits else pd.DataFrame([{"dataset": "no_loaded_data"}])
market_column_availability

## Connectedness input inspection

In [ ]:
connectedness_rows = []
for name, df in datasets.items():
    result = inspect_connectedness_inputs(df)
    connectedness_rows.append({"dataset": name, **result})

connectedness_inspection = pd.DataFrame(connectedness_rows) if connectedness_rows else pd.DataFrame([{"dataset": "no_loaded_data", "has_minimum_inputs": False}])
print(describe_gfevd_requirements())
connectedness_inspection

## Audit figures

In [ ]:
# Always save the source row-count audit because it reflects real file availability.
fig, ax = plt.subplots(figsize=(8, 4.5))
plot_df = file_inventory.copy()
plot_df["rows"] = [len(datasets.get(name, [])) if name in datasets else 0 for name in plot_df["file_name"]]
colors = plot_df["status"].map({"found": "#2563eb", "missing": "#9ca3af"}).fillna("#9ca3af")
ax.bar(plot_df["file_name"], plot_df["rows"], color=colors)
ax.set_title("Audit Records by Source")
ax.set_ylabel("Rows loaded")
ax.tick_params(axis="x", rotation=25)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "audit_records_by_source.png", dpi=160, bbox_inches="tight")
plt.close(fig)

created_figures = ["reports/figures/audit_records_by_source.png"]

if not standardized_sentiment.empty:
    sentiment_cols = [col for col in standardized_sentiment.columns if "sentiment" in col]
    if sentiment_cols:
        col = sentiment_cols[0]
        fig, ax = plt.subplots(figsize=(7, 4))
        if pd.api.types.is_numeric_dtype(standardized_sentiment[col]):
            standardized_sentiment[col].dropna().hist(ax=ax, bins=20, color="#2563eb")
            ax.set_xlabel(col)
        else:
            standardized_sentiment[col].fillna("missing").value_counts().plot(kind="bar", ax=ax, color="#2563eb")
        ax.set_title("Sentiment Distribution")
        ax.spines[["top", "right"]].set_visible(False)
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "sentiment_distribution.png", dpi=160, bbox_inches="tight")
        plt.close(fig)
        created_figures.append("reports/figures/sentiment_distribution.png")

records_source = None
for candidate in [standardized_sentiment, scraped_news, merged_data]:
    if not candidate.empty:
        date_cols = [col for col in candidate.columns if "date" in str(col).lower() or "time" in str(col).lower()]
        if date_cols:
            records_source = candidate.copy()
            records_source["audit_date"] = pd.to_datetime(records_source[date_cols[0]], errors="coerce").dt.date
            break

if records_source is not None:
    counts = records_source.groupby("audit_date", dropna=True).size()
    if not counts.empty:
        fig, ax = plt.subplots(figsize=(8, 4))
        counts.plot(ax=ax, marker="o", color="#2563eb")
        ax.set_title("Records Over Time")
        ax.set_ylabel("Record count")
        ax.spines[["top", "right"]].set_visible(False)
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "records_over_time.png", dpi=160, bbox_inches="tight")
        plt.close(fig)
        created_figures.append("reports/figures/records_over_time.png")

created_figures

## Current pipeline interpretation

The expected prototype appears to follow this planned sequence: sentiment test, scraping, stock filtering, keyword generation, daily sentiment aggregation, sentiment-market merging, and GFEVD analysis. Phase 0 preserves that expected structure, but the original raw files and scripts were not present in the workspace at audit time.

## Limitations

- Expected prototype files were not available in the current workspace.
- Legacy files are placeholders until the original scripts are added or recovered.
- No heavy scraping was run.
- No trading performance or recommendation claim is made.
- Connectedness analysis is not run in Phase 0.

## Next steps for Phase 1

- Add or recover `scraped_news.csv`, `sentiment_log.csv`, and `merged_data.json` if available.
- Clean sentiment logs and standardize date/company fields.
- Merge sentiment with market data.
- Validate numeric inputs for GFEVD/connectedness analysis.
- Repair or run connectedness analysis with clear limitations.